# v8 Cut 2b — the A100 PROBE (do the existing kernels change verdict on Ampere?)

Cut 2a measured on T4: **WMMA tensor cores are SLOWER than the CUDA-core GEMV at decode** (M=G≤16 too small to amortize the opaque-fragment tax). Before writing the hard `mma.m16n8k16`+cp.async kernel, this probe runs the kernels we ALREADY have — **Cut 1 (`v8_gqa`, CUDA core) and Cut 2a (`v8_gqa_tc`, WMMA)** — on a rented **A100 (sm_80)**, which has 5× bigger tensor cores + 6× the HBM BW. Two questions for ~10 min of rental:
1. **Does the tc-vs-CUDA-core verdict FLIP on Ampere?** If WMMA is still slower, the negative result is definitive and we skip the hard kernel. If it's competitive, THEN cp.async/mma tuning is worth it.
2. **Validate the Cut 1 `AI=2G/b` roofline on the actual A100** it was recorded for (Cut 1 was measured on T4).

vast.ai notes: torch is in a venv; `!`-cells need the venv `bin/` on PATH (handled below). Build now targets **sm_75 + sm_80** so the same kernels run on T4 and A100.

## 0. Dependencies + GPU (vast.ai venv-safe)

In [ ]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

# 1) Physical GPU on this runtime? (Colab defaults to CPU; pick a GPU explicitly.)
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit(
        'No CUDA GPU visible. On vast.ai pick an A100 instance; on Colab use Runtime > Change runtime '
        'type > GPU. (Cut 2b needs an A100 / sm_80.)')

# 2) Install deps (incl. numpy) BEFORE importing torch, so torch's numpy bridge initializes --
#    importing torch first on a numpy-less venv (vast.ai) prints 'Failed to initialize NumPy'.
pip('ninja', 'pytest', 'numpy')

# 3) torch present AND CUDA-enabled? A CPU-only wheel raises "not compiled with CUDA" on any kernel.
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False

if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit(
        'A GPU is present but torch was a CPU-only build -- installed the CUDA build. NOW: '
        'restart the kernel/session, then re-run this cell (the old CPU torch stays loaded until restart).')

# vast.ai/venv: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!which python && python -c "import torch; print('shell python sees torch', torch.__version__)"

## 1. Get the repo

In [ ]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

## 2. Roofline on the ACTUAL A100 (sm_80) — validate the Cut 1 `AI=2G/b` prediction here

In [ ]:
from roofline.archs import get_arch
from roofline.model import estimate
arch = get_arch('sm_80')
print('arch:', arch.name, '| HBM', arch.hbm_bw_gbps, 'GB/s | fp16 ridge',
      round(arch.fp16_tc_flops/(arch.hbm_bw_gbps*1e9),1))
print(f"{'G':>3} | {'AI=2G/b':>8} | {'limiter':>7} | {'t_hbm floor':>12}")
for G in (1,2,4,8,16,32):
    e = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='fp16', G=G)
    print(f'{G:>3} | {e.arithmetic_intensity:8.1f} | {e.limiter.upper():>7} | {e.t_hbm*1e3:9.4f}ms')


## 3. Build BOTH kernels for sm_80 (Cut 1 CUDA-core + Cut 2a WMMA)

In [ ]:
import glob, os, shutil
for nm in ('fa_v8_gqa', 'fa_v8_gqa_tc'):
    for d in glob.glob(os.path.expanduser(f'~/.cache/torch_extensions/*/{nm}')):
        if not glob.glob(os.path.join(d, '*.so')):
            shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
print('v8_gqa   :', build_kernel('v8_gqa'))
print('v8_gqa_tc:', build_kernel('v8_gqa_tc'))


## 4. Correctness sanity on A100 (quick — both backends already 64/64 + 38/38 on T4)

In [ ]:
!python -m pytest tests/test_correctness.py -k "v8_gqa_tc" -q


## 5. THE A/B on A100 — does WMMA beat the CUDA-core GEMV here (it lost on T4)?

In [ ]:
print('=== Cut 2a: v8_gqa_tc (WMMA tensor cores) on A100 ===')
!python -m bench.harness --backend v8_gqa_tc --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32
print('\n=== Cut 1: v8_gqa (CUDA cores) on A100 — same workload ===')
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32


## 6. Reclaim-SDPA-at-batch on A100 (both backends, G=8)

In [ ]:
print('=== v8_gqa_tc (WMMA) ===')
!python -m bench.harness --backend v8_gqa_tc --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
print('\n=== v8_gqa (CUDA core) ===')
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
